# Fine-tune LLM with Ray on SageMaker HyperPod (Studio Workflow)

**FSDP + LoRA on a Ray cluster managed through SageMaker Studio**

This notebook uses the new **Ray on HyperPod** integration in SageMaker Studio. Unlike the manual
approach (writing YAML manifests, installing Helm charts, debugging kubectl), Studio handles cluster
creation, Ray Dashboard access, Grafana observability, and workspace connectivity.

### What You Do vs. What Studio Does

| You | Studio |
|-----|--------|
| Write training script (`train_ray.py`) | Creates Ray cluster on HyperPod EKS |
| Configure hyperparameters (`args.yaml`) | Installs KubeRay operator and CRDs |
| Prepare dataset | Creates namespace, service account, storage |
| Submit and monitor job | Provides Ray Dashboard and Managed Grafana |
| | Connects JupyterLab/Code Editor to cluster |
| | Detects hung jobs |

### Prerequisites

1. A **SageMaker HyperPod** cluster with EKS orchestration
2. **SageMaker Studio** domain with access to the HyperPod cluster
3. **FSx for Lustre** file system attached to the cluster
4. This notebook running in a **Studio JupyterLab or Code Editor** space connected to the HyperPod cluster

## Step 1: Create a Ray Cluster from SageMaker Studio

1. Open **SageMaker Studio**
2. Navigate to **HyperPod clusters** in the left sidebar
3. Select your HyperPod cluster
4. Click **Create Ray cluster** and configure:
   - **Ray version**: 2.56.1
   - **Head node**: 1x instance (no GPU needed)
   - **Worker group**: 8x `g5.16xlarge` (1 GPU each, 8 total)
   - **Autoscaling**: min 1, max 16 workers
   - **Storage**: Select the FSx for Lustre file system
5. Click **Create**

Studio will:
- Install the KubeRay operator if not present
- Register RayCluster, RayService, RayJob CRDs
- Create the namespace, service account, and FSx PV/PVC
- Deploy the RayCluster with your configuration
- Validate that the cluster has sufficient pod capacity

Wait for the cluster status to show **Active** before continuing.

## Step 2: Connect Your Workspace

1. In Studio, click **Open Ray Dashboard** on your cluster — it opens directly (no `kubectl port-forward`)
2. Click **Open JupyterLab** or **Open Code Editor** to launch a workspace connected to the cluster
3. Your workspace has:
   - Direct access to the Ray cluster (Ray client auto-configured)
   - FSx for Lustre mounted at `/mnt/fsx`
   - Ray CLI available (`ray job submit`, `ray status`)

Open this notebook in the connected workspace to continue.

## Step 3: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Verify Ray cluster connectivity
import ray

ray.init()
print(f"Ray version: {ray.__version__}")
print(f"Available resources: {ray.cluster_resources()}")
print(f"Nodes: {len(ray.nodes())}")
ray.shutdown()

## Step 4: Prepare the Dataset

Since FSx for Lustre is mounted directly at `/mnt/fsx`, we write data there — no `kubectl cp` needed.

In [ ]:
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen3-0.6B"
DATASET_ID = "FreedomIntelligence/medical-o1-reasoning-SFT"
NUM_SAMPLES = 1000

dataset = load_dataset(DATASET_ID, "en", split=f"train[:{NUM_SAMPLES}]")
print(f"Dataset loaded: {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")
print(f"\nExample:\n{dataset[0]}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

SYSTEM_PROMPT = """You are a medical reasoning assistant. Respond to user queries
with clear, step-by-step medical reasoning."""


def format_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["Question"]},
        {
            "role": "assistant",
            "content": f"<think>\n{example['Complex_CoT']}\n</think>\n\n{example['Response']}",
        },
    ]
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return example


dataset = dataset.map(format_chat)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
import os

# Write directly to FSx — no kubectl cp needed
TRAIN_PATH = "/mnt/fsx/datasets/medical-o1/train"
VAL_PATH = "/mnt/fsx/datasets/medical-o1/val"

os.makedirs(TRAIN_PATH, exist_ok=True)
os.makedirs(VAL_PATH, exist_ok=True)

train_dataset.to_json(os.path.join(TRAIN_PATH, "train.json"))
val_dataset.to_json(os.path.join(VAL_PATH, "val.json"))

print(f"Train data written to: {TRAIN_PATH}")
print(f"Val data written to:   {VAL_PATH}")

## Step 5: Copy Training Script and Config to FSx

Ray workers access files from FSx. Copy the training script and config there.

In [ ]:
import shutil

SCRIPTS_PATH = "/mnt/fsx/scripts"
os.makedirs(SCRIPTS_PATH, exist_ok=True)

shutil.copy("train_ray.py", os.path.join(SCRIPTS_PATH, "train_ray.py"))
shutil.copy("args.yaml", os.path.join(SCRIPTS_PATH, "args.yaml"))
shutil.copy("requirements.txt", os.path.join(SCRIPTS_PATH, "requirements.txt"))

print(f"Files copied to {SCRIPTS_PATH}:")
print(os.listdir(SCRIPTS_PATH))

## Step 6: Submit the Training Job

Two options:
- **Option A**: Ray Jobs API (programmatic, recommended for notebooks)
- **Option B**: Ray CLI (quick, recommended for terminal)

### Option A: Ray Jobs API

In [ ]:
from ray.job_submission import JobSubmissionClient

# Studio auto-configures RAY_ADDRESS when your workspace is connected to the cluster.
# If not set, the default is the local cluster.
RAY_ADDRESS = os.environ.get("RAY_ADDRESS", "http://localhost:8265")

client = JobSubmissionClient(RAY_ADDRESS)
print(f"Connected to Ray at: {RAY_ADDRESS}")
print(f"Existing jobs: {len(client.list_jobs())}")

In [ ]:
job_id = client.submit_job(
    entrypoint="python /mnt/fsx/scripts/train_ray.py --config /mnt/fsx/scripts/args.yaml",
    runtime_env={
        "working_dir": "/mnt/fsx/scripts",
        "pip": "/mnt/fsx/scripts/requirements.txt",
        "env_vars": {
            "HF_HOME": "/mnt/fsx/huggingface",
            "NCCL_DEBUG": "INFO",
            "NCCL_SOCKET_IFNAME": "eth0",
        },
    },
    metadata={
        "name": "qwen3-medical-finetune",
        "cluster": "hyperpod-studio",
    },
)

print(f"Job submitted: {job_id}")
print(f"\nView in Ray Dashboard: click 'Open Ray Dashboard' in Studio")
print(f"View in Grafana: click 'Open Grafana' in Studio")

### Option B: Ray CLI

Alternatively, submit from the terminal in your connected workspace:

```bash
ray job submit \
    --working-dir /mnt/fsx/scripts \
    --runtime-env-json '{"pip": "/mnt/fsx/scripts/requirements.txt", "env_vars": {"HF_HOME": "/mnt/fsx/huggingface", "NCCL_DEBUG": "INFO"}}' \
    -- python /mnt/fsx/scripts/train_ray.py --config /mnt/fsx/scripts/args.yaml
```

## Step 7: Monitor the Job

You have three ways to monitor, all accessible from Studio:

1. **Ray Dashboard** — click **Open Ray Dashboard** on your cluster in Studio. Shows job status, worker logs, resource utilization, and task timeline.
2. **Amazon Managed Grafana** — click **Open Grafana** in Studio. Pre-configured dashboards for GPU utilization, memory, network, and training throughput.
3. **Programmatic** — poll from this notebook (below).

Studio also provides **hung job detection** — if your job stops making progress, you'll get an alert.

In [ ]:
import time

while True:
    status = client.get_job_status(job_id)
    print(f"Status: {status}")
    if status.is_terminal():
        break
    time.sleep(30)

logs = client.get_job_logs(job_id)
print(f"\n{'=' * 60}")
print(f"Job Logs (last 2000 chars):")
print(f"{'=' * 60}")
print(logs[-2000:])

## Step 8: Verify Output

Model artifacts are on FSx — directly accessible from this workspace.

In [ ]:
OUTPUT_DIR = "/mnt/fsx/output/qwen3-medical"

print("Output directory:")
if os.path.exists(OUTPUT_DIR):
    for item in os.listdir(OUTPUT_DIR):
        item_path = os.path.join(OUTPUT_DIR, item)
        if os.path.isdir(item_path):
            size = sum(f.stat().st_size for f in os.scandir(item_path) if f.is_file())
            print(f"  {item}/ ({size / 1e9:.2f} GB)")
        else:
            print(f"  {item} ({os.path.getsize(item_path) / 1e6:.1f} MB)")
else:
    print("  Output directory not found — job may still be running.")

In [ ]:
# Quick validation: load the merged model and run inference
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

MERGED_PATH = "/mnt/fsx/output/qwen3-medical/merged_model"

if os.path.exists(MERGED_PATH):
    print("Loading merged model for validation...")
    pipe = pipeline(
        "text-generation",
        model=MERGED_PATH,
        torch_dtype="auto",
        device_map="auto",
    )

    test_prompt = "What are the common symptoms and diagnostic criteria for Type 2 Diabetes?"
    result = pipe(
        [{"role": "user", "content": test_prompt}],
        max_new_tokens=512,
    )
    print(f"\nPrompt: {test_prompt}")
    print(f"\nResponse:\n{result[0]['generated_text'][-1]['content']}")
else:
    print(f"Merged model not found at {MERGED_PATH}")
    print("Check if merge_weights is enabled in args.yaml and job completed.")

## Step 9: Upload Model to S3 (Optional)

Copy the final model to S3 for deployment via SageMaker Endpoints or other services.

In [ ]:
import boto3

S3_BUCKET = "your-model-bucket"  # <-- Replace with your bucket
S3_PREFIX = "models/qwen3-medical-finetuned"

if os.path.exists(MERGED_PATH):
    s3 = boto3.client("s3")
    for root, dirs, files in os.walk(MERGED_PATH):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, MERGED_PATH)
            s3_key = f"{S3_PREFIX}/{relative_path}"
            print(f"Uploading: {relative_path}")
            s3.upload_file(local_path, S3_BUCKET, s3_key)

    print(f"\nModel uploaded to: s3://{S3_BUCKET}/{S3_PREFIX}/")
else:
    print("Merged model not found. Skipping S3 upload.")

## Step 10: Clean Up

When done, manage the Ray cluster from Studio:

1. **Scale down** — In Studio, select the cluster and reduce worker count to 0
2. **Delete** — In Studio, select the cluster and click **Delete**

HyperPod nodes remain available for other workloads after the Ray cluster is deleted.

In [ ]:
# Optionally clean up dataset and script files from FSx

# import shutil
# shutil.rmtree("/mnt/fsx/datasets/medical-o1", ignore_errors=True)
# shutil.rmtree("/mnt/fsx/scripts", ignore_errors=True)
# print("Cleaned up FSx working files.")

print("Cleanup commands above are commented out for safety. Uncomment to execute.")

## Summary

| Step | What happened |
|------|---------------|
| 1 | Created Ray cluster from SageMaker Studio UI |
| 2 | Connected JupyterLab workspace to the cluster |
| 3 | Installed Python dependencies |
| 4 | Prepared dataset and wrote to FSx |
| 5 | Copied training script to FSx |
| 6 | Submitted Ray training job |
| 7 | Monitored via Ray Dashboard and Managed Grafana (from Studio) |
| 8 | Verified model output on FSx |
| 9 | Uploaded model to S3 |
| 10 | Cleaned up Ray cluster from Studio |

### What We Didn't Have to Do

- Install or configure `kubectl` or `helm`
- Write Kubernetes YAML manifests (RayCluster, PV, PVC, Namespace, ServiceAccount)
- Install CRDs (KubeRay operator, Kubeflow Training Operator)
- Debug pod scheduling or node capacity issues
- Set up `kubectl port-forward` for the Ray Dashboard
- Configure Prometheus and Grafana for observability
- Manage Docker images for dependency changes
- Use `kubectl cp` or `kubectl exec` to move files and install packages